In [7]:
import os
import numpy as np
import pandas as pd
from prettytable import PrettyTable
from scipy.stats import chi2_contingency
from statsmodels.sandbox.regression.predstd import wls_prediction_std
from statsmodels.stats.proportion import proportions_ztest
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import warnings
import palettable.wesanderson as pal
from fpdf import FPDF

warnings.filterwarnings('ignore')

In [6]:
!pip install fpdf

  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=2cd15558fe46ed51b12980924532aff624718a81df35e62b834dbeb08c659eb9
  Stored in directory: /home/jovyan/.cache/pip/wheels/65/4f/66/bbda9866da446a72e206d6484cd97381cbc7859a7068541c36
Successfully built fpdf


In [8]:
subgroups = pd.read_csv('../Data_Prep/subgroup_meta.csv', index_col=0)
subgroups = subgroups.set_index('PATNO')

In [9]:
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.1)
    Q3 = df[column].quantile(0.9)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

class PDF(FPDF):

    def footer(self):
        self.set_y(-15)  # Set the position of the footer from the bottom of the page
        self.set_font('Arial', 'I', 8)  # Choose Arial italic 8
        self.cell(0, 10, 'Page %s' % self.page_no(), 0, 0, 'C')  # Add a cell for page numbers
    
    def header(self):
        self.set_font('Arial', 'B', 12)
        self.cell(0, 10, filename, 0, 1, 'C')

    def chapter_title(self, title):
        self.set_font('Arial', 'B', 16)
        self.cell(0, 5, title, 0, 1, 'L')
        self.ln(5)

    def chapter_body(self, body):
        self.set_font('Arial', '', 12)
        self.multi_cell(0, 10, body)
        self.ln()

    def add_table_from_string(self, data_str):
        # Assumes that data_str is a well-structured tab output from statsmodels
        lines = data_str.split('\n')[:-3]
        page_width = self.w - 2 * self.l_margin
        col_width = page_width / 7  # Divide page width into four columns
        self.set_font("Arial", size=8)
        
        for i , line in enumerate(lines):
            # Split by oversized space groups mimicking column structure
            cells = line.split('  ')
            cells = [i for i in cells if i != '']
            if i in [0,1,11,13,22,27]:  # Chapter titles or similar
                self.cell(page_width, 4, line.strip(), 0, 1, 'L')
            elif i <= 10 or i >= 22:
                for cell in cells:
                    self.cell(col_width, 4, cell.strip(), border=0)
                self.ln()
            else : 
                if i == 12 : 
                    cells = [''] + cells
                if i in [18,19] : 
                    cells[0] = 'Slope:' + cells[0][11:]
                for cell in cells:
                    self.cell(col_width, 4, cell.strip(), border=0)
                self.ln()

    def add_plot(self, plot_filename):
        self.image(plot_filename, w=180, h=100)
        self.ln()

def format_as_contingency(df , event) : 
    # Create a pivot table to format data into a contingency table
    pivot_table = df.pivot(index='Group', columns=event, values='count')
    
    # Convert the pivot table to a PrettyTable
    pretty_table = PrettyTable()
    
    # Add column names (Assuming 'Worse' and 'Not Worse' are the columns after pivoting)
    pretty_table.field_names = ["Group", "Worse", "Not Worse"]
    
    # Add rows to the pretty table from pivot data
    for index, row in pivot_table.iterrows():
        # Row must be added as a list of elements
        pretty_table.add_row([index, row['Worse'], row['Not Worse']])
    
    # Print the PrettyTable
    return pretty_table

# Defined function to handle pairwise comparison
def pairwise_proportion_test(counts, nobs, alpha=0.05):
    group_labels = ['Group 0', 'Group 1', 'Group 2']
    num_groups = len(counts)
    
    # Applying Bonferroni correction
    adjusted_alpha = alpha / num_groups
    
    print(f"Using a Bonferroni-corrected alpha value of {adjusted_alpha} \n" )
    
    for i in range(num_groups):
        for j in range(i + 1, num_groups):
            stat, p_val = proportions_ztest([counts[i], counts[j]], [nobs[i], nobs[j]])
            print(f"Comparison: {group_labels[i]} vs {group_labels[j]} - z-stat: {stat}, p-value: {p_val}")
            if p_val < adjusted_alpha:
                print(f"Significant difference between {group_labels[i]} and {group_labels[j]} at the {adjusted_alpha:.4f} significance level. \n")
            else:
                print(f"No significant difference between {group_labels[i]} and {group_labels[j]} at the {adjusted_alpha:.4f} significance level. \n")

def plot_regression_with_ci(x, y, color, label):
    # Fit the regression model
    X = sm.add_constant(x)  # Adds a constant term to the predictor
    model = sm.OLS(y, X).fit()

    # Get prediction std deviation and intervals
    prstd, iv_l, iv_u = wls_prediction_std(model)

    # Plotting
    #plt.plot(x['trajectory'].values, y, 'o', color=color, label=f'{label}')
    extend = x['trajectory'].max() -0.75
    plt.plot(x['trajectory'].values*extend, model.predict(X), 'k-', color=color, label=f'{label}')
    #plt.fill_between(x['trajectory'], iv_l, iv_u, color=color, alpha=0.05)

def data_availability_plot(data) : 
    fig , (ax1,ax2) = plt.subplots(figsize=(10,6) , ncols=2)
    pd.DataFrame(data.groupby('Group').count().values  / data.groupby('Group').size().values.reshape(-1,1) , index=  [0,1,2] , columns=data.columns[:-1]).T.plot(kind='line' , ax=ax1)
    ax1.set_title('Data Availability')
    ax1.set_ylabel('Proportion of Data Available')
    ax1.set_xlabel('Visit Number')

    pd.DataFrame(data.groupby('Group').mean()).T.plot(kind='line' , ax=ax2)
    ax2.set_title('Subgroup Averages')
    ax2.set_ylabel('Average Score')
    ax2.set_xlabel('Visit Number')
    
    return fig

def data_process(data , events_filt , events_df,print_final_event=True) : 
    if print_final_event : 
        print(f'final event : {events_filt[-1]}')
    
    df_long = data.reset_index().melt(id_vars=['Group' ,'PATNO'], var_name='EVENT_ID', value_name='Status')
    
    df_long = df_long.dropna(subset= ['Status'])
    
    df_long = pd.merge(df_long , subgroups.reset_index() , left_on = ['PATNO' , 'EVENT_ID' , 'Group'],right_on =['PATNO' , 'EVENT_ID' , 'Group'] , how='outer')
    
    df_long = pd.merge(df_long , events_df , on =['PATNO' , 'EVENT_ID'] , how='outer')
    
    df_long = df_long[df_long['EVENT_ID'].isin(events_filt)]
    
    # Combined linear model, including interaction between predictor and group
    df_long['Group'] = pd.Categorical(df_long['Group'], categories=[0,1,2], ordered=True)

    return df_long

def ols_model_plots(model) : 
    fitted_vals = model.predict()
    residuals = model.resid
    
    fig, (ax1,ax2) = plt.subplots(figsize=(10, 6) , ncols=2)
    ax1.scatter(fitted_vals, residuals, alpha=0.5)
    ax1.axhline(0, color='red', linestyle='--')
    ax1.set_xlabel('Fitted Values')
    ax1.set_ylabel('Residuals')
    ax1.set_title('Residuals vs Fitted Values')
    
    sm.qqplot(residuals, line='s', ax=ax2)  # 's' indicates standardized line
    ax2.set_title('Q-Q Plot of Residuals')

    return fig

def gen_patient_class(df_long , model , data) : 
    # Group data by PatientID, and fit a regression model for each group
    slopes = {}
    
    for patient_id, group in df_long.groupby('PATNO'):
        x = group['trajectory']
        y = group['Status'] - model.params['Intercept'] #- (model.params['AGE_AT_VISIT']*group['AGE_AT_VISIT']) - (model.params['GENDER']*group['GENDER'])
        if len(x) < 2 :
            slopes[patient_id] = 0
        else : 
            slopes[patient_id] = np.polyfit(x, y, 1, full=False)[0] 
    
    threshold = ((model.params[3] + model.params[4]) + (model.params[3] + model.params[5]) + model.params[3])/3
    data['Class'] = ['Worse' if slopes[patient] > threshold else 'Not Worse' for patient in data.index]
    data['Slope'] = [slopes[patient] for patient in data.index]
    
    data['Class'] = pd.Categorical(data['Class'] , categories=['Worse' , 'Not Worse'] , ordered=True)
    data_test = data.groupby(['Group','Class'], observed=False)['Class'].value_counts().reset_index().sort_values(['Group' , 'Class'])['count'].to_numpy().reshape(-1,2)

    return data, data_test

def boxplot_regression_plot(df_long) : 
    
    # Creating a figure
    fig = plt.figure(figsize=(12, 6))

    # This loop will handle the plotting
    for idx, group in enumerate(df_long['Group'].unique()):
        # Filter the dataframe by group and event
        sub_df = df_long[df_long['Group'] == group]
        plot_regression_with_ci(sub_df[['trajectory']], sub_df[['Status']], colors[idx], label=f'Group {idx}')
        # Create boxplots for each event within this group
        for event_id in event_ids:
            # Extract the specific group and event data
            group_event_data = sub_df[sub_df['EVENT_ID'] == event_id]['Status']
            
            # Calculate the position offset based on event_id
            position = np.where(event_ids == event_id)[0][0] * (len(df_long['Group'].unique()) + 1) + idx
            
            # Plotting the boxplot
            plt.boxplot(group_event_data, positions=[position], widths=0.6, patch_artist=True, boxprops=dict(facecolor=colors[idx],alpha=0.5))
        
    # Customizing the plot
    plt.xticks(range(1,len(event_ids)*4, 4), event_ids)
    plt.xlabel('Event ID')
    plt.ylabel('Status')
    plt.title('Grouped Boxplots of Status by Event ID and Group')
    plt.grid(True)
    
    # Adding legend manually
    handles = [plt.Line2D([0], [0], color=color, marker='s', linestyle='', markersize=10) for color in colors]
    plt.legend(handles, ['Group ' + str(i) for i in range(len(colors))], title="Group")
    
    return fig

In [10]:
events = ['BL', 'V02' , 'V04' ,'V05',
          'V06' ,'V07' , 'V08' ,'V09' , 'V10' ,
          'V11' , 'V12' ,'V13' , 'V14' ,'V15' ,
          'V16' , 'V17' , 'V18','V19' ,'V20' ]

all_events = ['SC', 'BL', 'V01', 'V02', 'V03', 'V04', 'V05',
'V06', 'V07', 'V08', 'V09', 'V10', 'V11', 'V12',
'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19',
'ST', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25'
]

x_list = pd.DataFrame(np.arange(0 , 9.5 , 0.5) , index = events , columns=['trajectory'])
events_df = pd.DataFrame()
for patno in subgroups.index.unique() : 
    df_tmp = pd.DataFrame({'PATNO' : [patno for i in range(len(events))] , 'EVENT_ID' : events})
    events_df = pd.concat([events_df , df_tmp])

events_df = pd.merge(events_df , x_list , left_on = 'EVENT_ID' ,right_index=True )

In [16]:
gen_pdf = True
if gen_pdf : 
    # Assume filename and other variables are defined
    pdf = PDF()

significant_check = []
filename_list = []
feature_list = []
page_number = []
r_squared = []
ref_group = []
m_ref = []
m_test1 = []
ci_test1 = []
pval_test1 = []
m_test2 = []
ci_test2 = []
pval_test2 = []
for page_i , filename in enumerate([file for file in sorted(os.listdir('./../Labels/')) if 'Scores' in file]) : 
    print(filename)
    check = False
    significant = False
    df = pd.read_csv(f'./../Labels/{filename}' , index_col=0)
    data = pd.merge(df , subgroups.reset_index()[['PATNO' , 'Group']].drop_duplicates() , left_index=True , right_on='PATNO').set_index('PATNO')

    fig_availability = data_availability_plot(data)
    fig_availability.savefig(f'data_availability_{page_i}.png' , dpi=64)
    plt.close()

    events_filt = sorted(list(set(events) & set(data.columns)))
    df_long = data_process(data , events_filt , events_df , print_final_event=False)

    df_long['Status'] = np.sqrt(df_long['Status'])
    # Combined linear model, including interaction between predictor and group
    df_long['Group'] = pd.Categorical(df_long['Group'], categories=[0,1,2], ordered=True)
    model1 = smf.ols('Status ~  AGE_AT_VISIT + GENDER', data=df_long.dropna()).fit()
    df_long['Status'] = model1.resid
    model1 = smf.ols('Status ~  trajectory * C(Group)', data=df_long.dropna()).fit()
    df_long['Group'] = pd.Categorical(df_long['Group'], categories=[2,1,0], ordered=True)
    model2 = smf.ols('Status ~  AGE_AT_VISIT + GENDER', data=df_long.dropna()).fit()
    df_long['Status'] = model2.resid
    model2 = smf.ols('Status ~  trajectory * C(Group)', data=df_long.dropna()).fit()
    
    if (sum(model1.pvalues.iloc[-2:].values < 0.05/3) > 0) & (model1.rsquared > 0.1) :
        model_summary = str(model1.summary())
        r2 = model1.rsquared
        reference = ['Group 0']
        params = model1.params
        ci = model1.conf_int()
        pvals = model1.pvalues
        
        fig_ols = ols_model_plots(model1)
        fig_ols.savefig(f'ols_{page_i}.png' , dpi=64)
        plt.close()

        df_long['Status'] = df_long['Status'].fillna(model1.predict(df_long))
        data, data_test = gen_patient_class(df_long , model1 , data)
        check = True
        significant = True
        
    elif (sum(model2.pvalues.iloc[-2:].values < 0.05/3) > 0) & (model2.rsquared > 0.1) :
        model_summary = str(model2.summary())
        r2 = model2.rsquared
        reference = ['Group 2']
        params = model2.params
        ci = model2.conf_int()
        pvals = model2.pvalues
        
        fig_ols = ols_model_plots(model2)
        fig_ols.savefig(f'ols_{page_i}.png' , dpi=64)
        plt.close()

        df_long['Status'] = df_long['Status'].fillna(model2.predict(df_long))
        data, data_test = gen_patient_class(df_long , model2 , data)
        check = True
        significant = True

    else : 
        model_summary = str(model1.summary())
        r2 = model1.rsquared
        reference = ['Group 0']
        params = model1.params
        ci = model1.conf_int()
        pvals = model1.pvalues
        
        fig_ols = ols_model_plots(model1)
        fig_ols.savefig(f'ols_{page_i}.png' , dpi=64)
        plt.close()

        df_long['Status'] = df_long['Status'].fillna(model1.predict(df_long))
        data, data_test = gen_patient_class(df_long , model1 , data)
        check = True
        significant = False

    data = data[events_filt + list(data.columns[-3:])]

    # Unique event IDs
    event_ids = events_df['EVENT_ID'].unique()
    
    # Colors for the groups
    colors = pal.FantasticFox2_5.hex_colors[:3]

    fig_boxplot = boxplot_regression_plot(df_long)
    fig_boxplot.savefig(f'boxplot_{page_i}.png' , dpi=64)
    plt.close()

    if gen_pdf : 
        pdf.add_page()

        # Adding sections
        pdf.chapter_title('Data Availability and Group Averages')
        pdf.add_plot(f'data_availability_{page_i}.png')
        
        pdf.chapter_title('Model Summary')
        pdf.add_table_from_string(model_summary)

        pdf.add_page()
        pdf.chapter_title('Model Performance Plots')
        pdf.add_plot(f'ols_{page_i}.png')
        
        pdf.chapter_title('Regression Boxplots')
        pdf.add_plot(f'boxplot_{page_i}.png')

    significant_check.append(significant)
    filename_list.append(filename.split('_')[0])
    feature_list.append(filename.split('_')[1])
    page_number.append((page_i*2) +1)
    r_squared.append(np.round(r2 , 3))
    ref_group.append(reference[0])
    m_ref.append(np.round(params[3] , 2))
    m_test1.append(np.round(params[4] , 2))
    ci_test1.append(np.round(ci.iloc[4].values , 2))
    pval_test1.append(np.round(pvals[4] , 3))
    m_test2.append(np.round(params[5] , 2))
    ci_test2.append(np.round(ci.iloc[5].values , 2))
    pval_test2.append(np.round(pvals[5] , 3))

    os.remove(f'data_availability_{page_i}.png')
    os.remove(f'ols_{page_i}.png')
    os.remove(f'boxplot_{page_i}.png')
        
if gen_pdf : 
    # Output the PDF
    pdf_output = f"ClinicalMeasuresReport.pdf"
    pdf.output(pdf_output)



MDS-UPDRS_NP3TOT_Scores.csv


In [48]:
table_1 = ['MDS-UPDRS_NP3TOT_Scores.csv' , 'QOLLOWERExtremity_Total_Scores.csv', 'QOLUPPERExtremity_Total_Scores.csv',  'SchwabEngland_MSEADLG_Scores.csv', 'MotorFunction_Total_Scores.csv',
            'MOCA_MCATOT_Scores.csv','HOPKINSVERBAL_HVLTTOT_Scores.csv', 'TrailMaking_Total_Scores.csv', 'LNSeq_LNS_TOTRAW_Scores.csv', 'SymbolDigit_SDMTOTAL_Scores.csv',  'SemanticFluency_Total_Scores.csv', 'BostonNaming_Total_Scores.csv',  'LexFL_LEXFLTOT_Scores.csv',
          'SCOPA_Total_Scores.csv',
            'Anxiety_Total_Scores.csv',
            'EPWORTH_ESSTOT_Scores.csv',  'REM_Total_Scores.csv',
            'MDS-UPDRS_NP1TOT_Scores.csv',  'MDS-UPDRS_NP2PTOT_Scores.csv',  'QOLCOMMUNICATION_Total_Scores.csv',  'QOLCOGNITION_Total_Scores.csv',
          ]

table_3 = [['MDS_UPDRS_P1_Factor_1' , 'MDS_UPDRS_P3_Factor_1' , 'MDS_UPDRS_P3_Factor_3' , 'MDS_UPDRS_P3_Factor_5' ] ,
          ['MDS_UPDRS_P1_Factor_1' , 'MDS_UPDRS_P3_Factor_1' , 'MDS_UPDRS_P3_Factor_4' , 'MDS_UPDRS_P3_Factor_5' ],
          ['MDS_UPDRS_P1_Factor_2' , 'MDS_UPDRS_P1_Factor_3' , 'MDS_UPDRS_P2_Factor_1' , 'MDS_UPDRS_P3_Factor_3' , 'MDS_UPDRS_P3_Factor_5' ],
          ['MDS_UPDRS_P1_Factor_1' , 'MDS_UPDRS_P1_Factor_3' , 'MDS_UPDRS_P3_Factor_3' , 'MDS_UPDRS_P3_Factor_5' ]]

table_4 = []

In [37]:
filename

'QOLCOGNITION_Total_Scores.csv'

In [11]:
gen_pdf = True
if gen_pdf : 
    # Assume filename and other variables are defined
    pdf = PDF()

significant_check = []
filename_list = []
feature_list = []
r_squared = []
slope_0 = []
slope_1 = []
slope_2 = []
m_pval_0_1 = []
m_pval_0_2 = []
m_pval_2_1 = []
c_0 = []
c_1 = []
c_2 = []
c_pval_0_1 = []
c_pval_0_2 = []
c_pval_2_1 = []

#for page_i , filename in enumerate([file for file in sorted(os.listdir('./../Labels/')) if 'Scores' in file]) : 
df_fa = pd.read_csv('./../Labels/FA_SCOPA.csv' , index_col = 0)
for page_i , filename in enumerate([col for col in df_fa.columns if col not in ['PATNO' , 'EVENT_ID']]): 
#for page_i , filename in enumerate(table_1) : 
    print(filename)
    check = False
    #df = pd.read_csv(f'./../Labels/{filename}' , index_col=0)
    #df_fa['combo'] = df_fa.loc[: , filename].sum(axis=1)
    df = df_fa.pivot_table(values = filename, columns = 'EVENT_ID' , index = 'PATNO' , observed=False)
    data = pd.merge(df , subgroups.reset_index()[['PATNO' , 'Group']].drop_duplicates() , left_index=True , right_on='PATNO').set_index('PATNO')

    events_filt = sorted(list(set(events) & set(data.columns)))
    df_long = data_process(data , events_filt , events_df , print_final_event=False)

    print((df_long['Status'] < 0 ).sum())
    
    #df_long['Status'] = np.sqrt(df_long['Status'])
    # Combined linear model, including interaction between predictor and group
    df_long['Group'] = pd.Categorical(df_long['Group'], categories=[0,1,2], ordered=True)
    model1 = smf.ols('Status ~  AGE_AT_VISIT + GENDER', data=df_long.dropna()).fit()
    df_long['Status'] = model1.resid
    model1 = smf.ols('Status ~  trajectory * C(Group)', data=df_long.dropna()).fit()
    df_long['Group'] = pd.Categorical(df_long['Group'], categories=[2,1,0], ordered=True)
    model2 = smf.ols('Status ~  AGE_AT_VISIT + GENDER', data=df_long.dropna()).fit()
    df_long['Status'] = model2.resid
    model2 = smf.ols('Status ~  trajectory * C(Group)', data=df_long.dropna()).fit()

    r2 = max(model1.rsquared , model2.rsquared)

    model_summary = str(model1.summary())

    params1 = model1.params
    ci1 = model1.params - model1.conf_int().iloc[:,0]

    model_summary = str(model2.summary())
    params2 = model2.params
    ci2 = model2.params - model2.conf_int().iloc[:,0]

    #filename_list.append('MDS-UPDRRS')
    filename_list.append(filename.split('_')[0])
    #feature_list.append('_'.join(filename.split('_')[2:]))
    feature_list.append(filename.split('_')[1])
    r_squared.append(np.round(r2 , 3))
    slope_0.append(f"{params1[3]:.2f} ± {ci1[3]:.2f}")
    slope_1.append(f"{params1[4]+params1[3]:.2f} ± {ci1[4]:.2f}")
    slope_2.append(f"{params1[5]+params1[3]:.2f} ± {ci1[5]:.2f}")
    m_pval_0_1.append(np.round(model1.pvalues[4] , 3))
    m_pval_0_2.append(np.round(model1.pvalues[5] , 3))
    m_pval_2_1.append(np.round(model2.pvalues[4] , 3))
    c_0.append(f"{params1[0]:.2f} ± {ci1[0]:.2f}")
    c_1.append(f"{params1[1]+params1[0]:.2f} ± {ci1[1]:.2f}")
    c_2.append(f"{params1[2]+params1[0]:.2f} ± {ci1[2]:.2f}")
    c_pval_0_1.append(np.round(model1.pvalues[1] , 3))
    c_pval_0_2.append(np.round(model1.pvalues[2] , 3))
    c_pval_2_1.append(np.round(model2.pvalues[1] , 3))




SCOPA_Factor_1
2626
SCOPA_Factor_2
3394
SCOPA_Factor_3
3499
SCOPA_Factor_4
3319
SCOPA_Factor_5
2746


In [12]:
output = pd.DataFrame({'Clinical Measure' : filename_list , 'Feature' : feature_list , 'R-Squared' : r_squared , 
                      'Slope Group 0' : slope_0 , 'Slope Group 1' : slope_1 , 'P-Val 0_1' : m_pval_0_1,
                       'Slope Group 2' : slope_2 , 'P-Val 0_2' : m_pval_0_2 , 
                       'P-val 1_2' : m_pval_2_1,
                       'Int Group 0' : c_0 , 'Int Group 1' : c_1 , 'C P-Val 0_1' : c_pval_0_1,
                       'Int Group 2' : c_2 , 'C P-Val 0_2' : c_pval_0_2 , 
                       'C P-val 1_2' : c_pval_2_1
                      })

In [13]:
output

,Clinical Measure,Feature,R-Squared,Slope Group 0,Slope Group 1,P-Val 0_1,Slope Group 2,P-Val 0_2,P-val 1_2,Int Group 0,Int Group 1,C P-Val 0_1,Int Group 2,C P-Val 0_2,C P-val 1_2
0,SCOPA,Factor,0.025,0.05 ± 0.02,0.05 ± 0.03,0.720,0.07 ± 0.03,0.134,0.152,-0.27 ± 0.09,-0.14 ± 0.11,0.022,-0.25 ± 0.12,0.748,0.031
1,SCOPA,Factor,0.010,0.02 ± 0.02,0.03 ± 0.03,0.284,0.05 ± 0.03,0.083,0.350,-0.18 ± 0.11,-0.12 ± 0.13,0.375,-0.10 ± 0.14,0.273,0.717
2,SCOPA,Factor,0.016,0.05 ± 0.02,0.03 ± 0.03,0.093,0.07 ± 0.03,0.379,0.003,-0.13 ± 0.10,-0.16 ± 0.12,0.617,-0.18 ± 0.13,0.428,0.675
3,SCOPA,Factor,0.016,0.03 ± 0.03,0.08 ± 0.03,0.007,0.03 ± 0.03,0.933,0.002,-0.11 ± 0.11,-0.25 ± 0.13,0.029,-0.14 ± 0.15,0.629,0.066
4,SCOPA,Factor,0.007,0.05 ± 0.02,0.01 ± 0.03,0.011,0.04 ± 0.03,0.709,0.019,-0.23 ± 0.10,0.01 ± 0.12,0.000,-0.16 ± 0.14,0.256,0.004


In [14]:
output.iloc[: , [1,5,7,8]]

,Feature,P-Val 0_1,P-Val 0_2,P-val 1_2
0,Factor,0.720,0.134,0.152
1,Factor,0.284,0.083,0.350
2,Factor,0.093,0.379,0.003
3,Factor,0.007,0.933,0.002
4,Factor,0.011,0.709,0.019


In [15]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 5.8 MB/s eta 0:00:00a 0:00:01


In [16]:
import openpyxl

In [17]:
output.to_csv('./../Visualisation/Table4.csv')
output.to_excel('Table4.xlsx')

In [19]:
output[np.logical_and(output['Clinical Measure'] == 'MDS-UPDRS' , output['Feature'] == 'NP3TOT')]

,Clinical Measure,Feature,Significant Difference,Page,R-Squared,Reference Group,Slope Reference Group,Slope Diff Ref vs Group 1,Confidence Interval 1,P-Value 1,Slope Diff Ref vs Group 0/2,Confidence Interval 0/2,P-Value 0/2
133,MDS-UPDRS,NP3TOT,True,267,0.152,Group 0,0.15,0.05,"[0.02, 0.08]",0.002,0.02,"[-0.01, 0.06]",0.158


In [17]:
output[np.logical_and(np.logical_or(output['P-Value 1'] < 0.05/3 , output['P-Value 0/2'] < 0.05/3) , output['R-Squared'] > 0.01)].sort_values('R-Squared' , ascending=False)

,Clinical Measure,Feature,Significant Difference,Page,R-Squared,Reference Group,Slope Reference Group,Slope Diff Ref vs Group 1,Confidence Interval 1,P-Value 1,Slope Diff Ref vs Group 0/2,Confidence Interval 0/2,P-Value 0/2
147,MDS-UPDRS,NP4TOT,True,295,0.264,Group 0,0.20,0.05,"[0.02, 0.08]",0.002,0.06,"[0.02, 0.09]",0.001
141,MDS-UPDRS,NP4FLCTI,True,283,0.220,Group 0,0.10,0.03,"[0.01, 0.04]",0.002,0.03,"[0.01, 0.04]",0.007
142,MDS-UPDRS,NP4FLCTX,True,285,0.204,Group 0,0.08,0.03,"[0.02, 0.05]",0.000,0.03,"[0.01, 0.05]",0.001
151,MDS-UPDRS,NP4WDYSK,True,303,0.161,Group 0,0.07,0.01,"[-0.01, 0.02]",0.265,0.02,"[0.01, 0.04]",0.008
133,MDS-UPDRS,NP3TOT,True,267,0.152,Group 0,0.15,0.05,"[0.02, 0.08]",0.002,0.02,"[-0.01, 0.06]",0.158
186,MotorFunction,SHUFFLE,True,373,0.112,Group 2,0.07,-0.03,"[-0.05, -0.01]",0.007,0.00,"[-0.02, 0.02]",0.995
185,MotorFunction,POORBAL,True,371,0.109,Group 0,0.07,-0.03,"[-0.06, -0.01]",0.004,-0.01,"[-0.03, 0.02]",0.578
150,MDS-UPDRS,NP4WDYSKPCT,True,301,0.108,Group 2,0.38,-0.17,"[-0.29, -0.06]",0.003,-0.13,"[-0.26, 0.01]",0.063
148,MDS-UPDRS,NP4WDYSKDEN,True,297,0.108,Group 2,0.15,-0.07,"[-0.11, -0.02]",0.004,-0.05,"[-0.1, 0.0]",0.061
212,QOLLOWERExtremity,Total,True,425,0.105,Group 2,-0.07,0.02,"[0.01, 0.04]",0.010,0.01,"[-0.01, 0.04]",0.206


In [14]:
output[output['Significant Difference']]

,Clinical Measure,Feature,Significant Difference,Page,R-Squared,Reference Group,Slope Reference Group,Slope Diff Ref vs Group 1,Confidence Interval 1,P-Value 1,Slope Diff Ref vs Group 0/2,Confidence Interval 0/2,P-Value 0/2
133,MDS-UPDRS,NP3TOT,True,267,0.152,Group 0,0.15,0.05,"[0.02, 0.08]",0.002,0.02,"[-0.01, 0.06]",0.158
141,MDS-UPDRS,NP4FLCTI,True,283,0.220,Group 0,0.10,0.03,"[0.01, 0.04]",0.002,0.03,"[0.01, 0.04]",0.007
142,MDS-UPDRS,NP4FLCTX,True,285,0.204,Group 0,0.08,0.03,"[0.02, 0.05]",0.000,0.03,"[0.01, 0.05]",0.001
147,MDS-UPDRS,NP4TOT,True,295,0.264,Group 0,0.20,0.05,"[0.02, 0.08]",0.002,0.06,"[0.02, 0.09]",0.001
148,MDS-UPDRS,NP4WDYSKDEN,True,297,0.108,Group 2,0.15,-0.07,"[-0.11, -0.02]",0.004,-0.05,"[-0.1, 0.0]",0.061
150,MDS-UPDRS,NP4WDYSKPCT,True,301,0.108,Group 2,0.38,-0.17,"[-0.29, -0.06]",0.003,-0.13,"[-0.26, 0.01]",0.063
151,MDS-UPDRS,NP4WDYSK,True,303,0.161,Group 0,0.07,0.01,"[-0.01, 0.02]",0.265,0.02,"[0.01, 0.04]",0.008
185,MotorFunction,POORBAL,True,371,0.109,Group 0,0.07,-0.03,"[-0.06, -0.01]",0.004,-0.01,"[-0.03, 0.02]",0.578
186,MotorFunction,SHUFFLE,True,373,0.112,Group 2,0.07,-0.03,"[-0.05, -0.01]",0.007,0.00,"[-0.02, 0.02]",0.995
212,QOLLOWERExtremity,Total,True,425,0.105,Group 2,-0.07,0.02,"[0.01, 0.04]",0.010,0.01,"[-0.01, 0.04]",0.206
